# Laboratorio 8 – Task 2
**CC3045 – Inteligencia Artificial**

Problema: Asignar 8 microservicios (M1–M8) a 3 servidores (S1, S2, S3) respetando restricciones de capacidad y anti-afinidad.

## Definición del Problema (CSP)

In [16]:
import random
import time
import copy

# Variables del CSP: los 8 microservicios
VARIABLES = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']

# Dominio de cada variable: los 3 servidores disponibles
DOMAIN = ['S1', 'S2', 'S3']

# Pares que no pueden estar en el mismo servidor (anti-afinidad)
ANTI_AFFINITY = [
    ('M1', 'M2'),
    ('M3', 'M4'),
    ('M5', 'M6'),
    ('M1', 'M5'),
]

# Máximo de microservicios permitidos por servidor
MAX_CAPACITY = 3

# Cuenta cuántas restricciones viola una asignación
# Usada como heurística en Beam Search y Local Search
def count_violations(assignment):
    violations = 0

    # Revisar pares de anti-afinidad que ya están asignados
    for (a, b) in ANTI_AFFINITY:
        if a in assignment and b in assignment:
            if assignment[a] == assignment[b]:
                violations += 1

    # Penalizar servidores que superen la capacidad máxima
    server_counts = {}
    for server in assignment.values():
        server_counts[server] = server_counts.get(server, 0) + 1
    for count in server_counts.values():
        if count > MAX_CAPACITY:
            violations += (count - MAX_CAPACITY)

    return violations

# Retorna True solo si la asignación está completa y sin violaciones
def is_valid(assignment):
    return len(assignment) == len(VARIABLES) and count_violations(assignment) == 0

# Muestra la asignación agrupada por servidor
def print_assignment(assignment, label='Asignacion'):
    print(f"\n{label}:")
    for server in DOMAIN:
        microservices = [m for m, s in assignment.items() if s == server]
        print(f"  {server}: {microservices}")
    print(f"  Violaciones: {count_violations(assignment)} | Valida: {is_valid(assignment)}")

print("Problema CSP definido correctamente.")
print(f"Variables: {VARIABLES}")
print(f"Dominio: {DOMAIN}")
print(f"Restricciones anti-afinidad: {ANTI_AFFINITY}")
print(f"Capacidad maxima por servidor: {MAX_CAPACITY}")

Problema CSP definido correctamente.
Variables: ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8']
Dominio: ['S1', 'S2', 'S3']
Restricciones anti-afinidad: [('M1', 'M2'), ('M3', 'M4'), ('M5', 'M6'), ('M1', 'M5')]
Capacidad maxima por servidor: 3


## Task 2.1 - Implementación de Backtracking Search

In [17]:
# Verifica si asignar `value` a `var` no rompe ninguna restricción activa
def is_consistent(var, value, assignment):
    # Revisar restricciones de anti-afinidad contra variables ya asignadas
    for (a, b) in ANTI_AFFINITY:
        if a == var and b in assignment:
            if assignment[b] == value:
                return False
        if b == var and a in assignment:
            if assignment[a] == value:
                return False

    # Verificar que el servidor no supere la capacidad
    current_count = sum(1 for s in assignment.values() if s == value)
    if current_count >= MAX_CAPACITY:
        return False

    return True

# Reduce los dominios de variables vecinas tras asignar `value` a `var`
# Retorna None si algún dominio queda vacío (backtrack inmediato)
def forward_checking(var, value, assignment, domains):
    new_domains = copy.deepcopy(domains)

    # Eliminar `value` del dominio de vecinos por anti-afinidad
    for (a, b) in ANTI_AFFINITY:
        neighbor = None
        if a == var:
            neighbor = b
        elif b == var:
            neighbor = a

        if neighbor and neighbor not in assignment:
            if value in new_domains[neighbor]:
                new_domains[neighbor].remove(value)
            # Dominio vacío -> backtrack inmediato
            if not new_domains[neighbor]:
                return None

    # Si un servidor alcanzó capacidad máxima, quitarlo del dominio de los no asignados
    temp_assignment = {**assignment, var: value}
    server_counts = {}
    for s in temp_assignment.values():
        server_counts[s] = server_counts.get(s, 0) + 1

    for server, count in server_counts.items():
        if count >= MAX_CAPACITY:
            for unassigned in VARIABLES:
                if unassigned not in temp_assignment:
                    if server in new_domains[unassigned]:
                        new_domains[unassigned].remove(server)
                    if not new_domains[unassigned]:
                        return None

    return new_domains

# Backtracking recursivo con Forward Checking
def backtracking_search(assignment, domains):
    # Caso base: todas las variables tienen valor asignado
    if len(assignment) == len(VARIABLES):
        return assignment

    # Seleccionar la siguiente variable sin asignar
    unassigned = [v for v in VARIABLES if v not in assignment]
    var = unassigned[0]

    for value in domains[var]:
        if is_consistent(var, value, assignment):
            # Aplicar Forward Checking antes de profundizar
            new_domains = forward_checking(var, value, assignment, domains)

            if new_domains is not None:
                assignment[var] = value
                result = backtracking_search(assignment, new_domains)
                if result is not None:
                    return result
                # Sin solución en esta rama, deshacer la asignación
                del assignment[var]

    return None

# Ejecutar Backtracking Search
initial_domains = {var: list(DOMAIN) for var in VARIABLES}

start = time.time()
solution_bt = backtracking_search({}, initial_domains)
time_bt = time.time() - start

print("Resultados Backtracking Search")
if solution_bt:
    print_assignment(solution_bt, "Solucion encontrada")
else:
    print("No se encontro solucion.")
print(f"\nTiempo de ejecucion: {time_bt:.9f} segundos")

Resultados Backtracking Search

Solucion encontrada:
  S1: ['M1', 'M3', 'M6']
  S2: ['M2', 'M4', 'M5']
  S3: ['M7', 'M8']
  Violaciones: 0 | Valida: True

Tiempo de ejecucion: 0.000000000 segundos


### Análisis Task 2.1

El algoritmo encontró solución en la primera exploración válida, sin necesidad de retroceder: `S1: [M1, M3, M6]`, `S2: [M2, M4, M5]`, `S3: [M7, M8]`.

Esto se explica porque Forward Checking actúa antes de profundizar: al asignar M1 a S1, elimina S1 del dominio de M2 y M5 inmediatamente, reduciendo el espacio que el algoritmo necesita explorar. El resultado respeta todas las restricciones: ningún par de anti-afinidad comparte servidor y ningún servidor supera los 3 microservicios. El tiempo de ejecución fue prácticamente cero, lo cual era esperable dado que el árbol se poda de forma muy agresiva desde el inicio.

## Task 2.2 - Implementación de Beam Search

In [18]:
# Beam Search: expande estados nivel a nivel y conserva los K mejores
# La heurística de selección es count_violations (menos viola, mejor)
def beam_search(K=3):
    # Estado inicial: asignación vacía
    beam = [{}]

    for var in VARIABLES:
        candidates = []

        # Expandir cada estado del beam con todos los valores posibles
        for state in beam:
            for value in DOMAIN:
                new_state = {**state, var: value}
                candidates.append(new_state)

        # Ordenar por violaciones y conservar solo los K mejores (prune)
        candidates.sort(key=lambda s: count_violations(s))
        beam = candidates[:K]

    # El mejor candidato del beam al terminar todos los niveles
    return beam[0] if beam else None

# Probar con diferentes valores de K para ver el efecto
print("Resultados Beam Search")
for K in [1, 3, 6]:
    start = time.time()
    result = beam_search(K=K)
    elapsed = time.time() - start

    if result:
        print_assignment(result, f"Beam Search K={K}")
    else:
        print(f"Beam Search K={K}: No se encontro solucion.")
    print(f"  Tiempo: {elapsed:.9f} segundos")

# Guardar resultado con K=3 para benchmarking
start = time.time()
solution_beam = beam_search(K=3)
time_beam = time.time() - start

Resultados Beam Search

Beam Search K=1:
  S1: ['M1', 'M3', 'M6']
  S2: ['M2', 'M4', 'M5']
  S3: ['M7', 'M8']
  Violaciones: 0 | Valida: True
  Tiempo: 0.000000000 segundos

Beam Search K=3:
  S1: ['M1', 'M3', 'M6']
  S2: ['M2', 'M4', 'M5']
  S3: ['M7', 'M8']
  Violaciones: 0 | Valida: True
  Tiempo: 0.000000000 segundos

Beam Search K=6:
  S1: ['M1', 'M3', 'M6']
  S2: ['M2', 'M4', 'M5']
  S3: ['M7', 'M8']
  Violaciones: 0 | Valida: True
  Tiempo: 0.000000000 segundos


### Análisis Task 2.2

Los tres valores de K (1, 3 y 6) llegaron exactamente a la misma solución: `S1: [M1, M3, M6]`, `S2: [M2, M4, M5]`, `S3: [M7, M8]`, con 0 violaciones. Esto indica que en este problema la solución válida es tan dominante desde el primer nivel que incluso el beam más restrictivo (K=1) la conserva sin descartarla.

El tiempo fue idéntico y despreciable en los tres casos. En problemas más grandes o con restricciones más cruzadas, K=1 sería riesgoso porque una mala decisión temprana no tiene corrección posible; K=3 o K=6 darían más margen. Aquí, la estructura del problema favorece a Beam Search independientemente del K elegido.

## Task 2.3 - Implementación de Local Search (ICM)

In [19]:
# Genera una asignación completamente aleatoria como punto de partida
def random_assignment():
    return {var: random.choice(DOMAIN) for var in VARIABLES}

# Local Search mediante Modos Condicionales Iterados (ICM)
# En cada iteración recorre todas las variables y mueve cada una
# al servidor que reduzca más las violaciones globales
def local_search_icm(max_iterations=500):
    # Punto de partida aleatorio, probablemente con violaciones
    current = random_assignment()
    print(f"Asignacion inicial – Violaciones: {count_violations(current)}")

    best_seen = copy.deepcopy(current)
    best_violations = count_violations(current)

    for iteration in range(max_iterations):
        current_violations = count_violations(current)

        # Parar si ya no hay violaciones
        if current_violations == 0:
            break

        improved = False

        # Para cada variable, elegir el valor que minimice las violaciones
        for var in VARIABLES:
            best_value = current[var]
            best_local = current_violations

            for value in DOMAIN:
                candidate = {**current, var: value}
                v = count_violations(candidate)
                if v < best_local:
                    best_local = v
                    best_value = value

            # Aplicar el cambio si mejora el estado
            if best_value != current[var]:
                current[var] = best_value
                improved = True

        # Guardar el mejor estado global encontrado hasta ahora
        current_v = count_violations(current)
        if current_v < best_violations:
            best_violations = current_v
            best_seen = copy.deepcopy(current)

        # Si ninguna variable mejoró, estamos en un óptimo local
        if not improved:
            print(f"Optimo local en iteracion {iteration + 1} – Violaciones: {count_violations(current)}")
            break

    return best_seen

# Ejecutar Local Search ICM
random.seed(42)
start = time.time()
solution_icm = local_search_icm(max_iterations=500)
time_icm = time.time() - start

print("\nResultados Local Search ICM")
print_assignment(solution_icm, "Mejor solucion encontrada")
print(f"Tiempo de ejecucion: {time_icm:.9f} segundos")

Asignacion inicial – Violaciones: 2

Resultados Local Search ICM

Mejor solucion encontrada:
  S1: ['M6', 'M7', 'M8']
  S2: ['M2', 'M3', 'M5']
  S3: ['M1', 'M4']
  Violaciones: 0 | Valida: True
Tiempo de ejecucion: 0.000000000 segundos


### Análisis Task 2.3

El algoritmo arrancó con una asignación aleatoria que ya tenía 2 violaciones y en pocas iteraciones logró eliminarlas todas. La solución final fue `S1: [M6, M7, M8]`, `S2: [M2, M3, M5]`, `S3: [M1, M4]`, distinta a la encontrada por Backtracking y Beam Search, lo que demuestra que el CSP tiene múltiples soluciones válidas y que ICM converge a una de ellas dependiendo del punto de partida.

No se reportó un óptimo local, lo que significa que el algoritmo pudo mejorar en cada pasada hasta llegar a 0 violaciones sin quedarse atascado. Esto fue favorable dado el punto de partida, pero con una semilla diferente el resultado podría haber sido distinto.